# Digital Pathology Assistant — Train & Evaluate

This notebook puts **you** in charge of the full loop and explains each step:

1. Set up the environment and check for a GPU.
2. Choose a **data source** — synthetic (no files needed) or your **real annotated tiles**.
3. Train the tile classifier (uses the GPU automatically if present).
4. **Understand the results**: training curves, confusion matrix, metrics, sample predictions.
5. Run the full decision-support pipeline on a slide and view the heatmap + report.
6. Save a **CPU-loadable** checkpoint you can run anywhere.

> The model here learns a *toy* signal unless you supply real data. Nothing in this notebook is clinically validated.

## 1. Setup

On **Colab**: set `Runtime → Change runtime type → GPU` first. Then get the code — either clone your repo or upload the project folder. Adjust the path below.

In [ ]:
import os, sys

# macOS only: PyTorch and matplotlib each bundle an OpenMP runtime. When both
# load, libomp aborts the process (you'd see a macOS crash dialog). This env var
# permits the duplicate and is a no-op on Linux/Colab. Must be set before imports.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

# --- On Colab, uncomment ONE of these to get the code ---
# !git clone https://github.com/<you>/detection_hospital.git
# PROJECT_DIR = '/content/detection_hospital'

# When running locally from the notebooks/ folder, the project root is one level up:
PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('Project dir:', PROJECT_DIR)

In [ ]:
# Install dependencies. On Colab, torch+CUDA is already present, so this is quick.
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from pathassist.device import resolve_device

device = resolve_device('auto')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Training will use device:', device)

## 2. Choose your data

Set `DATA_SOURCE` to one of:

- `'synthetic'` — generated tiles, no files needed (good for a first run).
- `'folder'`  — your tiles in `DATA_DIR`, one subfolder per class, e.g.
  `data/tiles/benign/*.png` and `data/tiles/malignant/*.png`.
- `'csv'`     — a CSV (`DATA_CSV`) with `path,label` columns.

The cell also **exports a synthetic folder dataset** so you can try the real-data
loader path immediately, then swap in your own tiles later by editing these variables.

In [ ]:
from pathassist.synthetic import export_tile_folder
from pathassist.train import build_dataset

TILE_SIZE = 64

# ---- EDIT THIS to control where tiles come from ----
DATA_SOURCE = 'folder'          # 'synthetic' | 'folder' | 'csv'
DATA_DIR    = 'data/tiles'      # used when DATA_SOURCE == 'folder'
DATA_CSV    = 'data/labels.csv' # used when DATA_SOURCE == 'csv'
POSITIVE_LABELS = ['malignant'] # class names meaning positive (for CSV/named labels)
# ----------------------------------------------------

# For the demo, create a synthetic folder dataset if you don't have real tiles yet.
if DATA_SOURCE == 'folder' and not os.path.isdir(DATA_DIR):
    print('No real tiles found — exporting a synthetic demo dataset to', DATA_DIR)
    export_tile_folder(DATA_DIR, num_samples=600, tile_size=TILE_SIZE, seed=1)

tiles, labels = build_dataset(
    source=DATA_SOURCE,
    data_dir=DATA_DIR,
    data_csv=DATA_CSV,
    positive_labels=POSITIVE_LABELS,
    tile_size=TILE_SIZE,
    num_samples=600,
)
print(f'Loaded {len(tiles)} tiles | positives: {int(labels.sum())} | negatives: {int((labels==0).sum())}')

In [ ]:
# Sanity-check the data by eye: a few tiles from each class.
def show_examples(tiles, labels, per_class=4):
    fig, axes = plt.subplots(2, per_class, figsize=(3 * per_class, 6))
    for row, (label_value, name) in enumerate([(0.0, 'negative / benign'), (1.0, 'positive / malignant')]):
        idx = np.where(labels == label_value)[0][:per_class]
        for col in range(per_class):
            ax = axes[row, col]
            ax.axis('off')
            if col < len(idx):
                ax.imshow(tiles[idx[col]])
                ax.set_title(name if col == 0 else '')
    plt.tight_layout(); plt.show()

show_examples(tiles, labels)

## 3. Train

`train(...)` uses the GPU automatically (`device_preference='auto'`), saves a
**CPU-loadable** checkpoint, and returns a `TrainResult` with the history and
validation predictions we visualise below.

In [ ]:
from pathassist.train import train

result = train(
    epochs=12,
    batch_size=32,
    learning_rate=1e-3,
    device_preference='auto',
    version='0.1.0',
    out_path='models/tile_classifier.pt',
    tiles=tiles,
    labels=labels,
)
print('Checkpoint:', result.checkpoint_path)

## 4. Understand the results

In [ ]:
# Training curves: loss should fall; val accuracy should rise and plateau.
hist = result.history
epochs = [h['epoch'] for h in hist]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, [h['train_loss'] for h in hist], marker='o', label='train')
ax1.plot(epochs, [h['val_loss'] for h in hist], marker='o', label='val')
ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, [h['val_acc'] for h in hist], marker='o', color='green')
ax2.set_title('Validation accuracy'); ax2.set_xlabel('epoch'); ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix + core metrics on the validation set (threshold 0.5).
y_true = result.val_true.astype(int)
y_pred = (result.val_prob >= 0.5).astype(int)

tp = int(((y_pred == 1) & (y_true == 1)).sum())
tn = int(((y_pred == 0) & (y_true == 0)).sum())
fp = int(((y_pred == 1) & (y_true == 0)).sum())
fn = int(((y_pred == 0) & (y_true == 1)).sum())
cm = np.array([[tn, fp], [fn, tp]])

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall    = tp / (tp + fn) if (tp + fn) else 0.0   # sensitivity — critical in screening
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred neg', 'pred pos'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['true neg', 'true pos'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=16)
ax.set_title('Confusion matrix (val)')
plt.tight_layout(); plt.show()

print(f'Precision: {precision:.3f}')
print(f'Recall (sensitivity): {recall:.3f}   <- missed cancers live here')
print(f'F1: {f1:.3f}')

In [ ]:
# Score-distribution: how separable are the two classes? Well-separated peaks are good.
plt.figure(figsize=(8, 4))
plt.hist(result.val_prob[y_true == 0], bins=20, alpha=0.6, label='true negative')
plt.hist(result.val_prob[y_true == 1], bins=20, alpha=0.6, label='true positive')
plt.axvline(0.5, color='k', linestyle='--', label='threshold 0.5')
plt.xlabel('predicted malignancy probability'); plt.ylabel('count')
plt.legend(); plt.tight_layout(); plt.show()

## 5. Run the full pipeline on a slide

This uses the trained checkpoint through `TorchScorer` exactly as the CLI would —
tiling → scoring → triage → heatmap → report — and displays the explainability
overlay and the drafted report.

In [ ]:
from pathassist.config import load_config
from pathassist.synthetic import make_synthetic_slide
from pathassist.scoring import TorchScorer
from pathassist.pipeline import analyze_case
from pathassist.audit import AuditStore

config = load_config()
# Match tiling to the model's tile size for a coherent demo.
config['tiling']['tile_size'] = result.tile_size

slide = make_synthetic_slide(seed=7)  # replace with load_image('path/to/slide.png') for real slides
scorer = TorchScorer(str(result.checkpoint_path), device_preference='auto')
store = AuditStore(config['audit']['store_dir'])

case, heatmap_path, report_path = analyze_case(
    case_id='NB-001', image=slide, scorer=scorer,
    config=config, output_dir='outputs', audit_store=store,
)
print(f'Priority: {case.priority} | case score: {case.case_score:.2f} | '
      f'suspicious tiles: {case.suspicious_tile_count}/{case.tile_count}')

In [ ]:
from PIL import Image

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(slide); ax1.set_title('Slide'); ax1.axis('off')
ax2.imshow(Image.open(heatmap_path)); ax2.set_title('Malignancy heatmap (green→red)'); ax2.axis('off')
plt.tight_layout(); plt.show()

print(open(report_path).read())

## 6. Save / download the checkpoint

The checkpoint was saved on CPU, so it loads on any machine. On a laptop with no GPU:

```bash
python -m pathassist.cli run --case-id CASE-001 --image slide.png \
    --scorer torch --checkpoint models/tile_classifier.pt --device cpu
```

In [ ]:
# On Colab, download the trained checkpoint to your computer:
try:
    from google.colab import files
    files.download(str(result.checkpoint_path))
except Exception as e:
    print('Not on Colab (or download skipped). Checkpoint is at:', result.checkpoint_path)